# Experiment 01: three stage durations — run on the server

Run the cells in order.  Step 1 (simulation) takes a couple of minutes and runs inside the notebook.  Step 2 (TSC,
120 tasks) runs as a **background process** that writes to a log file: the notebook stays free, closing the browser
does not stop it, and the *Progress* cell can be re-run at any time.

If the session ends (24-hour limit) before step 2 is finished, open the notebook again and run *Settings*, then
*Step 2* and *Progress*: finished tasks are skipped, the run continues where it stopped.

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------------------
import os, sys, time, subprocess, glob, json
from pathlib import Path

EXP = Path.cwd()                      # this notebook lives in experiments/exp01_three_durations/
QUICK = False                         # True: small smoke test on data_quick/ (minutes instead of hours)
WORKERS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
MAX_ITER = 60 if QUICK else None      # cap on the flow-matching iterations: tests only
TIDES_DIR = None                      # folder containing tides_benchmarks.py, for the protocol check (optional)

sys.path.insert(0, str(EXP.parents[1]))
cfg = json.loads((EXP / "config.json").read_text())
DATA = EXP / (cfg["data_dir"] + ("_quick" if QUICK else ""))
LOG = EXP / ("run_tsc_quick.log" if QUICK else "run_tsc.log")
print(f"experiment folder {EXP}\ndata folder       {DATA}\nworkers           {WORKERS}")

In [ ]:
# ---- Environment check -------------------------------------------------------------------------------------------
import numpy as np, platform
print("python", platform.python_version(), "| numpy", np.__version__, "| cores available:", WORKERS)
try:
    mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2 ** 30
    print(f"memory: {mem:.0f} GB")
except (ValueError, OSError, AttributeError):
    pass

In [ ]:
# ---- Optional: the protocol is the TIDES structural benchmark ------------------------------------------------------
if TIDES_DIR:
    subprocess.run([sys.executable, "check_protocol.py", "--tides", str(TIDES_DIR)], cwd=EXP, check=True)
else:
    print("skipped (set TIDES_DIR to run it)")

## Step 1 — simulate all durations (a few minutes)
Writes the protocol, its truth, and the clean trajectories of every duration into the data folder.  Durations that
are already done are skipped.

In [ ]:
args = [sys.executable, "run_simulate.py"] + (["--quick"] if QUICK else [])
subprocess.run(args, cwd=EXP, check=True)

## Step 2 — TSC reconstructions in the background
Starts `run_tsc.py` with all workers as a separate process.  Re-running this cell while it is still running does
nothing; after it has stopped (finished, or the session ended), re-running it resumes the remaining tasks.

In [ ]:
PIDFILE = EXP / (".run_tsc_quick.pid" if QUICK else ".run_tsc.pid")

def tsc_running():
    """The pid of the background run started by this notebook, if it is still alive (None otherwise).
    Only the pid recorded at launch is trusted, and only if that process is still run_tsc.py."""
    if not PIDFILE.exists():
        return None
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, 0)
        cmd = Path(f"/proc/{pid}/cmdline").read_bytes().decode(errors="ignore") if Path(f"/proc/{pid}").exists() else "run_tsc.py"
        return pid if "run_tsc.py" in cmd else None
    except (ProcessLookupError, PermissionError):
        return None

if tsc_running():
    print("run_tsc.py is already running, pid", tsc_running())
else:
    args = [sys.executable, "-u", "run_tsc.py", "--workers", str(WORKERS)] + (["--quick"] if QUICK else []) \
           + (["--max-iter", str(MAX_ITER)] if MAX_ITER else [])
    with open(LOG, "a") as log:
        proc = subprocess.Popen(args, cwd=EXP, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    PIDFILE.write_text(str(proc.pid))
    print("started run_tsc.py, pid", proc.pid, "| log:", LOG)

## Progress — re-run at any time

In [ ]:
n_sets = 2 if QUICK else cfg["tsc"]["n_sets"]
total = len(cfg["durations"]) * len(cfg["tsc"]["k_values"]) * len(cfg["noise_levels"]) * n_sets
done = sorted(glob.glob(str(DATA / "tsc" / "*" / "*" / "*" / "set*.json")))
print(f"{len(done)} of {total} TSC tasks done | running: {bool(tsc_running())}")
for label in cfg["durations"]:
    row = []
    for k in cfg["tsc"]["k_values"]:
        n = len(glob.glob(str(DATA / "tsc" / label / "*" / f"k{k}" / "set*.json")))
        row.append(f"k={k}: {n:2d}")
    print(f"  {label:5s}  " + "   ".join(row))
if LOG.exists():
    print("\nlast lines of the log:")
    print("".join(LOG.read_text().splitlines(keepends=True)[-8:]))

## Summary — a first look once tasks are done

In [ ]:
subprocess.run([sys.executable, "inspect_tsc.py"] + (["--quick"] if QUICK else []), cwd=EXP, check=True)

## Stop the background run (only if needed)
Finished tasks are kept; a task interrupted half-way is simply recomputed next time.

In [ ]:
pid = tsc_running()
if pid:
    os.killpg(pid, 15)                 # the run was started in its own process group, whose id is its pid
    print("stopped run_tsc.py, pid", pid)
else:
    print("no background run started from this notebook is running")